# Portfolio Analysis and Risk Metrics

Computes full risk metrics (VaR, CVaR, Sharpe, Sortino, Calmar, drawdown) for both
an equal-weight and an optimised portfolio derived from the stocks in config.yaml.

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from src.utils.config import load_config
from src.analysis.portfolio import (
    calculate_portfolio_returns,
    calculate_cumulative_returns,
    calculate_correlation_matrix,
    calculate_covariance_matrix,
    minimum_variance_weights,
    maximum_sharpe_weights,
    portfolio_summary,
)
from src.analysis.risk_metrics import (
    calculate_var,
    calculate_cvar,
    calculate_sharpe_ratio,
    rolling_var,
    risk_summary,
)
from src.visualization.plots import (
    plot_portfolio_performance,
    plot_drawdown,
    plot_rolling_var,
    plot_efficient_frontier,
    plot_correlation_heatmap,
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

In [ ]:
cfg = load_config()
conf_level = cfg['risk_metrics']['var_confidence']
rf_rate    = 0.02

returns = pd.read_csv('../data/raw/stock_returns.csv', index_col='Date')
returns.index = pd.to_datetime(returns.index)
returns = returns.select_dtypes(include='number')

# Restrict to tickers that appear in config and exist in data
cfg_tickers = [t for t in cfg['stocks'] if t in returns.columns]
returns     = returns[cfg_tickers].dropna()

print(f'Tickers: {cfg_tickers}')
print(f'Rows:    {len(returns)}')
print(returns.tail(3))

In [ ]:
# Equal-weight portfolio
n = len(cfg_tickers)
equal_weights = np.ones(n) / n

eq_port_returns = calculate_portfolio_returns(returns, equal_weights)
eq_cum          = calculate_cumulative_returns(eq_port_returns)

print('Equal-weight portfolio summary:')
eq_summary = portfolio_summary(returns, equal_weights, risk_free_rate=rf_rate)
for k, v in eq_summary.items():
    print(f'  {k}: {v:.4f}')

In [ ]:
# Optimised portfolios
mv_weights     = minimum_variance_weights(returns)
sharpe_weights = maximum_sharpe_weights(returns, risk_free_rate=rf_rate)

print('Minimum Variance weights:')
for t, w in zip(cfg_tickers, mv_weights):
    print(f'  {t}: {w:.4f}')

print('\nMaximum Sharpe weights:')
for t, w in zip(cfg_tickers, sharpe_weights):
    print(f'  {t}: {w:.4f}')

In [ ]:
# Compare portfolio summaries
port_configs = [
    ('Equal Weight',     equal_weights),
    ('Min Variance',     mv_weights),
    ('Max Sharpe',       sharpe_weights),
]

rows = []
for name, w in port_configs:
    s = portfolio_summary(returns, w, risk_free_rate=rf_rate)
    s['portfolio'] = name
    rows.append(s)

comparison = pd.DataFrame(rows).set_index('portfolio')
print(comparison.round(4))

In [ ]:
# Risk metrics for equal-weight portfolio
print('Risk metrics (equal-weight):')
rs = risk_summary(eq_port_returns, confidence=conf_level, risk_free_rate=rf_rate)
for k, v in rs.items():
    print(f'  {k}: {v:.4f}')

In [ ]:
# Cumulative performance chart
fig, ax = plt.subplots(figsize=(13, 5))
for name, w in port_configs:
    pr = calculate_portfolio_returns(returns, w)
    cr = calculate_cumulative_returns(pr)
    ax.plot(cr.index, cr.values, label=name, linewidth=1.4)
ax.axhline(1.0, color='gray', linestyle='--', linewidth=0.8)
ax.set_title('Cumulative Portfolio Returns')
ax.set_xlabel('Date')
ax.set_ylabel('Cumulative Return (base 1.0)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig = plot_drawdown(eq_cum, title='Equal-Weight Portfolio Drawdown')
plt.show()

In [ ]:
# Rolling 30-day VaR
roll_v = rolling_var(eq_port_returns, window=30, confidence=conf_level)
fig = plot_rolling_var(roll_v, title=f'Rolling 30-Day VaR ({int(conf_level*100)}% CL)')
plt.show()

In [ ]:
# Efficient frontier
ann_ret = returns.mean() * 252
cov_ann = calculate_covariance_matrix(returns, annualize=True)
fig = plot_efficient_frontier(ann_ret, cov_ann, n_portfolios=5000, risk_free_rate=rf_rate)
plt.show()

In [ ]:
fig = plot_correlation_heatmap(calculate_correlation_matrix(returns))
plt.show()